In [1]:
!pip install -q groq ipywidgets PyPDF2


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from groq import Groq
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

API_KEY = "your_api_key"
client = Groq(api_key=API_KEY)

In [3]:
def explain_legal_term(jargon, level, tone):
    """Fetches the tailored explanation from the Groq AI API."""
    
    # We use f-strings to inject the user's choices directly into the prompt!
    prompt = f"""
    You are an expert legal assistant. Your task is to explain the following legal text.
    
    Target Audience Level: {level}
    Desired Tone/Format: {tone}
    
    Instructions:
    - If the Level is 'Beginner', use very simple analogies like you are talking to a child.
    - If the Level is 'Intermediate', explain it clearly but keep some professional context.
    - If the Level is 'Legal student', provide a deeper, technical breakdown of the legal mechanics.
    - If the Tone is 'Bullet points', strictly format your entire response as a bulleted list.
    - Always include a brief definition and a real-world example appropriate for the chosen level.
    
    Legal Text to Explain:
    {jargon}
    """
    
    try:
        # Asking the free Llama brain
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.1-8b-instant", 
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

In [4]:
import io
import PyPDF2
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# ==========================================
# 1. HEADER: Clean and Native
# ==========================================
# header = widgets.HTML("""
# <div style="background-color: #2563eb; padding: 25px; border-radius: 8px; color: white; text-align: center; margin-bottom: 15px; font-family: sans-serif;">
#     <h2 style="margin: 0; font-weight: 600;">⚖️ Legal Terms Explainer </h2>
#     <p style="margin: 5px 0 0 0; font-size: 15px; opacity: 0.9;">Simplifying the law with AI </p>
# </div>
# """)
header = widgets.HTML("""
<div style="background: linear-gradient(135deg, #2c3e50, #3498db); padding: 25px; border-radius: 8px 8px 0 0; color: white; text-align: center;">
    <h1 style="margin: 0; font-family: 'Segoe UI', sans-serif; font-weight: 600;">⚖️ Legal Terms Explainer</h1>
    <p style="margin: 5px 0 0 0; font-size: 16px; font-style: italic; opacity: 0.9;">Simplifying the law with AI.</p>
</div>
""")


# ==========================================
# 2. CONTROLS: Simple and Aligned
# ==========================================
input_box = widgets.Textarea(
    value='',
    placeholder='Paste legal contract text here, or upload a PDF document...',
    layout=widgets.Layout(width='100%', height='180px')
)

upload_button = widgets.FileUpload(
    accept='.pdf',
    multiple=False,
    description='Upload PDF',
    button_style='warning', # Standard yellow button
    layout=widgets.Layout(width='auto')
)

level_dropdown = widgets.Dropdown(
    options=['Beginner (Layman)', 'Intermediate (Business)', 'Advanced (Law Student)'],
    value='Beginner (Layman)',
    description='Target Audience:',
    style={'description_width': 'initial'} # Prevents text cutoff
)

tone_dropdown = widgets.Dropdown(
    options=['Standard Translation', 'In-Depth Analysis', 'Bullet Point Summary'],
    value='Standard Translation',
    description='Output Format:',
    style={'description_width': 'initial'}
)

button = widgets.Button(
    description='Explain It!',
    button_style='primary', # Standard blue button
    icon='check',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = widgets.Output(layout=widgets.Layout(
    width='100%', 
    padding='20px', 
    border='1px solid #d1d5db', # Light grey border
    background_color='#f9fafb', # Off-white background for readability
    border_radius='8px',
    margin='20px 0 0 0'
))

# ==========================================
# 3. LOGIC: The App Functions
# ==========================================
def on_file_upload(change):
    if upload_button.value:
        with output_area:
            clear_output()
            print("Reading your PDF... Please wait.")
            try:
                if isinstance(upload_button.value, tuple):
                    content = upload_button.value[0]['content'] 
                else: 
                    filename = list(upload_button.value.keys())[0]
                    content = upload_button.value[filename]['content']

                pdf_stream = io.BytesIO(content)
                reader = PyPDF2.PdfReader(pdf_stream)
                
                extracted_text = ""
                num_pages = min(len(reader.pages), 3) 
                for i in range(num_pages):
                    extracted_text += reader.pages[i].extract_text() + "\n\n"
                    
                input_box.value = extracted_text
                print(f"✅ Successfully pulled text from {num_pages} page(s)!")
            except Exception as e:
                print(f"❌ Error reading PDF: {e}")

upload_button.observe(on_file_upload, names='value')

def on_button_clicked(b):
    with output_area:
        if input_box.value.strip() == "":
            clear_output()
            print("Please enter or upload legal text first.")
            return
            
        clear_output()
        selected_level = level_dropdown.value
        selected_tone = tone_dropdown.value
        
        display(widgets.HTML(f"<i>Analyzing text for a {selected_level} audience...</i>"))
        
        explanation = explain_legal_term(input_box.value, selected_level, selected_tone) 
        
        clear_output()
        display(Markdown(explanation))

button.on_click(on_button_clicked)

# ==========================================
# 4. ASSEMBLE: The Layout
# ==========================================
# Grouping dropdowns together
dropdowns_row = widgets.HBox([level_dropdown, tone_dropdown], layout=widgets.Layout(grid_gap='20px', margin='10px 0'))

# Grouping buttons together
buttons_row = widgets.HBox([upload_button, button], layout=widgets.Layout(justify_content='space-between', margin='15px 0'))

# The Main Container
app_container = widgets.VBox([
    header, 
    input_box, 
    dropdowns_row, 
    buttons_row, 
    output_area
], layout=widgets.Layout(
    width='800px', 
    margin='20px auto', # Centers the app
    padding='25px', 
    border='1px solid #e5e7eb', 
    border_radius='12px', 
    background_color='#ffffff', 
    box_shadow='0 4px 6px -1px rgba(0, 0, 0, 0.1)' # Soft, clean shadow
))

display(app_container)